# Stage 3 CLIP Hybrid Train-Selected Clean (Kaggle)

CLIP coarse classifier with hyperparameters selected on train only, then evaluated once on clean val.


In [ ]:
import json
import shutil
import subprocess
import traceback
from pathlib import Path
from collections import Counter

RUN_NAME = "stage3_clip_hybrid_train_selected_clean"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
ARCHIVE_PATH = Path("/kaggle/working/stage3_deliverables_clip_hybrid_train_selected_clean.tar.gz")
LOG_PATH = WORK_DIR / "run_log.txt"
ERROR_PATH = WORK_DIR / "error_traceback.txt"

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
]
STAGE3_REL = Path("stage3_regrouped_v2")
TRAIN_JSONL_REL = STAGE3_REL / "train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl"
VAL_JSONL_REL = STAGE3_REL / "val/vlm_labels_v1_val_v2.annotated.jsonl"

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LABELS = ["insulator_ok", "defect_flashover", "defect_broken"]
SEED = 42
TEXT_PROMPTS = {
    "insulator_ok": "a clear photo crop of an intact power line insulator with regular shape and no visible damage",
    "defect_flashover": "a photo crop of a power line insulator with flashover burn marks dark surface traces or localized surface damage",
    "defect_broken": "a photo crop of a broken power line insulator with missing fragment damaged edge or structural break",
}

def log(msg):
    print(msg, flush=True)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

def sh(args, cwd: Path | None = None, check: bool = True):
    log("$ " + " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        log(p.stdout)
    if p.stderr:
        log(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {' '.join(str(a) for a in args)}")
    return p

def load_jsonl(path: Path):
    rows=[]
    with path.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def resolve_crop(row, split_root: Path) -> Path:
    p = Path(row["crop_path"])
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    candidates.extend([
        split_root / p,
        split_root.parent / p,
        split_root.parent / "crops" / row.get("split", "") / row.get("coarse_class", "") / p.name,
    ])
    for cand in candidates:
        if cand.exists():
            return cand
    raise FileNotFoundError(f"crop not found for {row.get('record_id')}: {row.get('crop_path')}")

def package_outputs():
    try:
        if ARCHIVE_PATH.exists():
            ARCHIVE_PATH.unlink()
        sh(["tar", "-czf", str(ARCHIVE_PATH), "-C", str(WORK_DIR.parent), RUN_NAME], check=False)
        log(f"Archive: {ARCHIVE_PATH}")
    except Exception:
        print(traceback.format_exc(), flush=True)

try:
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    log(f"RUN_NAME: {RUN_NAME}")
    sh(["nvidia-smi"], check=False)

    DATA_ROOT = None
    for root in DATASET_ROOT_CANDIDATES:
        if (root / TRAIN_JSONL_REL).exists() and (root / VAL_JSONL_REL).exists():
            DATA_ROOT = root
            break
    if DATA_ROOT is None:
        raise FileNotFoundError("Could not find stage3_regrouped_v2 train/val JSONL in Kaggle inputs")
    train_jsonl = DATA_ROOT / TRAIN_JSONL_REL
    val_jsonl = DATA_ROOT / VAL_JSONL_REL
    train_root = train_jsonl.parent
    val_root = val_jsonl.parent
    log(f"DATA_ROOT: {DATA_ROOT}")

    sh(["python", "-m", "pip", "install", "-q", "transformers==4.57.1", "scikit-learn", "tabulate"])

    import numpy as np
    import pandas as pd
    import torch
    from PIL import Image
    from sklearn.base import clone
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    from sklearn.model_selection import StratifiedKFold
    from sklearn.preprocessing import normalize
    from transformers import CLIPModel, CLIPProcessor

    train_rows = [r for r in load_jsonl(train_jsonl) if r.get("coarse_class") in LABELS]
    val_rows = [r for r in load_jsonl(val_jsonl) if r.get("coarse_class") in LABELS]
    log(f"train rows: {len(train_rows)} {Counter(r['coarse_class'] for r in train_rows)}")
    log(f"val rows: {len(val_rows)} {Counter(r['coarse_class'] for r in val_rows)}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"device: {device}")
    model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
    processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
    model.eval()

    def embed_images(rows, split_root: Path, batch_size: int = 32):
        feats=[]; labels=[]; ids=[]
        with torch.no_grad():
            for start in range(0, len(rows), batch_size):
                batch = rows[start:start+batch_size]
                images=[Image.open(resolve_crop(row, split_root)).convert("RGB") for row in batch]
                inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
                feats.append(model.get_image_features(**inputs).detach().cpu().numpy())
                labels.extend([r["coarse_class"] for r in batch])
                ids.extend([r["record_id"] for r in batch])
        return normalize(np.vstack(feats)), np.array(labels), ids

    def eval_pred(y_true, pred, prefix=""):
        return {
            prefix + "accuracy": accuracy_score(y_true, pred),
            prefix + "macro_f1_3class": f1_score(y_true, pred, labels=LABELS, average="macro", zero_division=0),
            prefix + "ok_recall": ((pred == "insulator_ok") & (y_true == "insulator_ok")).sum() / max((y_true == "insulator_ok").sum(), 1),
            prefix + "flashover_recall": ((pred == "defect_flashover") & (y_true == "defect_flashover")).sum() / max((y_true == "defect_flashover").sum(), 1),
            prefix + "broken_recall": ((pred == "defect_broken") & (y_true == "defect_broken")).sum() / max((y_true == "defect_broken").sum(), 1),
        }

    X_train, y_train, train_ids = embed_images(train_rows, train_root)
    X_val, y_val, val_ids = embed_images(val_rows, val_root)
    log(f"embeddings: {X_train.shape} {X_val.shape}")

    candidates=[]
    for class_weight in [None, "balanced"]:
        for C in [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]:
            candidates.append({"name": f"logreg_c{C:g}_{'balanced' if class_weight else 'plain'}", "C": C, "class_weight": class_weight})

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    cv_rows=[]
    for cand in candidates:
        fold_metrics=[]
        for fold, (tr_idx, dev_idx) in enumerate(cv.split(X_train, y_train), start=1):
            clf = LogisticRegression(max_iter=3000, C=cand["C"], class_weight=cand["class_weight"])
            clf.fit(X_train[tr_idx], y_train[tr_idx])
            pred = clf.predict(X_train[dev_idx])
            m = eval_pred(y_train[dev_idx], pred)
            m["fold"] = fold
            fold_metrics.append(m)
        row = {"method": cand["name"], "C": cand["C"], "class_weight": str(cand["class_weight"])}
        for key in ["accuracy", "macro_f1_3class", "ok_recall", "flashover_recall", "broken_recall"]:
            row["cv_" + key] = float(np.mean([m[key] for m in fold_metrics]))
        cv_rows.append(row)

    cv_summary = pd.DataFrame(cv_rows).sort_values(["cv_macro_f1_3class", "cv_accuracy"], ascending=False)
    cv_summary.to_csv(WORK_DIR / "train_cv_summary.csv", index=False)
    best = cv_summary.iloc[0].to_dict()
    log("Best train-CV method:")
    log(best)

    best_clf = LogisticRegression(max_iter=3000, C=float(best["C"]), class_weight=None if best["class_weight"] == "None" else best["class_weight"])
    best_clf.fit(X_train, y_train)
    val_pred = best_clf.predict(X_val)
    val_metrics = eval_pred(y_val, val_pred, prefix="val_")
    val_summary = pd.DataFrame([{**best, **val_metrics}])
    val_summary.to_csv(WORK_DIR / "val_summary_selected_by_train_cv.csv", index=False)

    pred_rows=[]
    for rid, gt, pred in zip(val_ids, y_val, val_pred):
        pred_rows.append({"record_id": rid, "gt": gt, "pred": pred, "correct": bool(gt == pred)})
    pd.DataFrame(pred_rows).to_csv(WORK_DIR / "val_predictions_selected_by_train_cv.csv", index=False)

    report = classification_report(y_val, val_pred, labels=LABELS, zero_division=0)
    cm = confusion_matrix(y_val, val_pred, labels=LABELS)
    (WORK_DIR / "val_classification_report.txt").write_text(report, encoding="utf-8")
    pd.DataFrame(cm, index=LABELS, columns=LABELS).to_csv(WORK_DIR / "val_confusion_matrix.csv")

    lines = [
        "# Stage 3 CLIP Hybrid Train-Selected Clean",
        "",
        "Hyperparameters were selected using 5-fold stratified cross-validation on `train_balanced` only. The clean val slice was evaluated once after selection.",
        "",
        "## Selected method",
        "",
        str(best),
        "",
        "## Val result",
        "",
        val_summary.to_markdown(index=False),
        "",
        "## Val classification report",
        "",
        "```text",
        report,
        "```",
        "",
        "## Top train-CV candidates",
        "",
        cv_summary.head(10).to_markdown(index=False),
    ]
    (WORK_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
    log(val_summary.to_string(index=False))
    package_outputs()
except Exception:
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    tb = traceback.format_exc()
    ERROR_PATH.write_text(tb, encoding="utf-8")
    print(tb, flush=True)
    package_outputs()
    raise
